In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vinaymandal/nifty50-dataset-2000-2026")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\vedant\.cache\kagglehub\datasets\vinaymandal\nifty50-dataset-2000-2026\versions\1


In [4]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.data_loader import load_nifty_data
from src.validation import validate_data, identify_extreme_returns

In [5]:
DATA_PATH = ROOT / "data" / "raw" / "nifty50.csv"

df = load_nifty_data(DATA_PATH)
df = df.sort_values("Date").reset_index(drop=True)
df.index

RangeIndex(start=0, stop=4585, step=1)

In [6]:
df.head()

,Date,Open,High,Low,Close
0,2007-09-17,4518.450195,4549.049805,4482.850098,4494.649902
1,2007-09-18,4494.100098,4551.799805,4481.549805,4546.200195
2,2007-09-19,4550.250000,4739.000000,4550.250000,4732.350098
3,2007-09-20,4734.850098,4760.850098,4721.149902,4747.549805
4,2007-09-21,4752.950195,4855.700195,4733.700195,4837.549805


In [7]:
report = validate_data(df)

report

{'rows': 4585,
 'date_validation': {'missing_dates': 0,
  'duplicate_dates': 0,
  'sorted': True,
  'min_date': Timestamp('2007-09-17 00:00:00'),
  'max_date': Timestamp('2026-05-29 00:00:00')},
 'ohlc_validation': {'missing_values': {'Open': 0,
   'High': 0,
   'Low': 0,
   'Close': 0},
  'non_positive_values': {'Open': 0, 'High': 0, 'Low': 0, 'Close': 0},
  'high_less_than_low': 0,
  'open_outside_range': 0,
  'close_outside_range': 0}}

In [8]:
df.describe()

,Date,Open,High,Low,Close
count,4585,4585.000000,4585.000000,4585.000000,4585.000000
mean,2017-01-28 14:53:31.341330,11211.588126,11270.584393,11137.269332,11205.282231
min,2007-09-17 00:00:00,2553.600098,2585.300049,2252.750000,2524.199951
25%,2012-05-25 00:00:00,5707.299805,5744.950195,5671.250000,5704.200195
50%,2017-02-07 00:00:00,8827.950195,8883.000000,8770.200195,8808.900391
75%,2021-10-05 00:00:00,16270.049805,16338.750000,16172.599609,16258.250000
max,2026-05-29 00:00:00,26333.699219,26373.199219,26210.050781,26328.550781
std,NaN,6572.792483,6593.037510,6546.850415,6570.466637


In [9]:
extreme = identify_extreme_returns(
    df, threshold=0.05
)

extreme

,Date,Open,High,Low,Close,DailyReturn
25,2007-10-23,5185.299805,5488.500000,5176.850098,5473.700195,0.055884
86,2008-01-21,5705.000000,5705.000000,4977.100098,5208.799805,-0.087024
87,2008-01-22,5203.350098,5203.350098,4448.500000,4899.299805,-0.059419
88,2008-01-23,4903.049805,5328.049805,4891.600098,5203.399902,0.062070
90,2008-01-25,5035.049805,5399.250000,5035.049805,5383.350098,0.069515
101,2008-02-11,5120.549805,5126.399902,4803.600098,4857.000000,-0.051432
104,2008-02-14,4944.649902,5220.250000,4944.649902,5202.000000,0.055290
116,2008-03-03,5222.799805,5222.799805,4936.049805,4953.000000,-0.051785
123,2008-03-13,4868.700195,4868.799805,4580.149902,4623.600098,-0.050985
125,2008-03-17,4745.450195,4745.450195,4482.100098,4503.100098,-0.051140


In [10]:
print("Rows:", len(df))
print("Start:", df["Date"].min())
print("End:", df["Date"].max())

Rows: 4585
Start: 2007-09-17 00:00:00
End: 2026-05-29 00:00:00


# 3. Event Detection

In [11]:
import importlib
import src.events

importlib.reload(src.events)

<module 'src.events' from 'c:\\Users\\vedant\\Projects\\algochowk-quant-research\\src\\events.py'>

In [12]:
import numpy as np
import pandas as pd

from src.events import run_event_study

In [13]:
df = (
    df
    .sort_values("Date")
    .reset_index(drop=True)
)

In [14]:
events = run_event_study(
    df,
    threshold=-0.03,
    holding_periods=(1, 3, 5, 10),
    exclude_overlapping=True,
    overlap_window=5,
)

In [15]:
events.shape

(53, 18)

In [16]:
events.columns.tolist()

['Date',
 'Open',
 'High',
 'Low',
 'Close',
 'event_return',
 'entry_date_1',
 'entry_price_1',
 'forward_return_1',
 'entry_date_3',
 'entry_price_3',
 'forward_return_3',
 'entry_date_5',
 'entry_price_5',
 'forward_return_5',
 'entry_date_10',
 'entry_price_10',
 'forward_return_10']

In [17]:
events["period"] = np.where(
    events["Date"] < pd.Timestamp("2020-01-01"),
    "development",
    "oos",
)

In [18]:
events["period"].value_counts()

period
development    38
oos            15
Name: count, dtype: int64

In [19]:
event_columns = ['Date',
 'Open',
 'High',
 'Low',
 'Close',
 'event_return',
 'entry_date_1',
 'entry_price_1',
 'forward_return_1',
 'entry_date_3',
 'entry_price_3',
 'forward_return_3',
 'entry_date_5',
 'entry_price_5',
 'forward_return_5',
 'entry_date_10',
 'entry_price_10',
 'forward_return_10',
 'period']

events[event_columns].head()

,Date,Open,High,Low,Close,event_return,entry_date_1,entry_price_1,forward_return_1,entry_date_3,entry_price_3,forward_return_3,entry_date_5,entry_price_5,forward_return_5,entry_date_10,entry_price_10,forward_return_10,period
0,2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000,-0.037469,2007-10-19,5360.350098,-0.027060,2007-10-19,5360.350098,0.021146,2007-10-19,5360.350098,0.038915,2007-10-19,5360.350098,0.094415,development
1,2007-11-21,5778.799805,5790.049805,5530.850098,5561.049805,-0.038030,2007-11-22,5564.649902,-0.008141,2007-11-22,5564.649902,0.030020,2007-11-22,5564.649902,0.009506,2007-11-22,5564.649902,0.067453,development
2,2007-12-17,6037.950195,6039.950195,5740.600098,5777.000000,-0.044761,2007-12-18,5777.600098,-0.006110,2007-12-18,5777.600098,-0.001921,2007-12-18,5777.600098,0.050739,2007-12-18,5777.600098,0.069544,development
3,2008-01-18,5907.750000,5908.750000,5677.000000,5705.299805,-0.035159,2008-01-21,5705.000000,-0.086976,2008-01-21,5705.000000,-0.087923,2008-01-21,5705.000000,-0.056380,2008-01-21,5705.000000,-0.067967,development
4,2008-02-07,5322.549805,5344.600098,5113.850098,5133.250000,-0.035566,2008-02-08,5132.100098,-0.002290,2008-02-08,5132.100098,-0.057257,2008-02-08,5132.100098,0.013620,2008-02-08,5132.100098,0.011633,development


In [20]:
events[event_columns].tail(10)

,Date,Open,High,Low,Close,event_return,entry_date_1,entry_price_1,forward_return_1,entry_date_3,entry_price_3,forward_return_3,entry_date_5,entry_price_5,forward_return_5,entry_date_10,entry_price_10,forward_return_10,period
43,2020-05-04,9533.500000,9533.500000,9266.950195,9293.500000,-0.057445,2020-05-05,9429.400391,-0.023734,2020-05-05,9429.400391,-0.024429,2020-05-05,9429.400391,-0.020171,2020-05-05,9429.400391,-0.064283,oos
44,2020-05-18,9158.299805,9158.299805,8806.750000,8823.250000,-0.034323,2020-05-19,8961.700195,-0.009217,2020-05-19,8961.700195,0.016130,2020-05-19,8961.700195,0.007515,2020-05-19,8961.700195,0.113527,oos
45,2020-12-21,13741.900391,13777.500000,13131.450195,13328.400391,-0.031405,2020-12-22,13373.650391,0.006928,2020-12-22,13373.650391,0.028085,2020-12-22,13373.650391,0.041795,2020-12-22,13373.650391,0.061752,oos
46,2021-02-26,14888.599609,14919.450195,14467.750000,14529.150391,-0.037636,2021-03-01,14702.500000,0.004016,2021-03-01,14702.500000,0.036939,2021-03-01,14702.500000,0.016024,2021-03-01,14702.500000,0.015440,oos
47,2021-04-12,14644.650391,14652.500000,14248.700195,14310.799805,-0.035326,2021-04-13,14364.900391,0.009739,2021-04-13,14364.900391,0.017609,2021-04-13,14364.900391,-0.004769,2021-04-13,14364.900391,0.034783,oos
48,2022-02-14,17076.150391,17099.500000,16809.650391,16842.800781,-0.030616,2022-02-15,16933.250000,0.024756,2022-02-15,16933.250000,0.021930,2022-02-15,16933.250000,0.016146,2022-02-15,16933.250000,-0.008229,oos
49,2022-02-24,16548.900391,16705.250000,16203.250000,16247.950195,-0.047781,2022-02-25,16515.650391,0.008643,2022-02-25,16515.650391,0.005467,2022-02-25,16515.650391,-0.016366,2022-02-25,16515.650391,0.006951,oos
50,2024-06-04,23179.500000,23179.500000,21281.449219,21884.500000,-0.059294,2024-06-05,22128.349609,0.022234,2024-06-05,22128.349609,0.052503,2024-06-05,22128.349609,0.051359,2024-06-05,22128.349609,0.062709,oos
51,2025-04-07,21758.400391,22254.000000,21743.650391,22161.599609,-0.032433,2025-04-08,22446.750000,0.003969,2025-04-08,22446.750000,0.017009,2025-04-08,22446.750000,0.044124,2025-04-08,22446.750000,0.080188,oos
52,2026-03-19,23197.750000,23378.699219,22930.349609,23002.150391,-0.032621,2026-03-20,23110.150391,0.000188,2026-03-20,23110.150391,-0.008557,2026-03-20,23110.150391,-0.012572,2026-03-20,23110.150391,0.000584,oos


In [21]:
events.head()

,Date,Open,High,Low,Close,event_return,entry_date_1,entry_price_1,forward_return_1,entry_date_3,entry_price_3,forward_return_3,entry_date_5,entry_price_5,forward_return_5,entry_date_10,entry_price_10,forward_return_10,period
0,2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000,-0.037469,2007-10-19,5360.350098,-0.027060,2007-10-19,5360.350098,0.021146,2007-10-19,5360.350098,0.038915,2007-10-19,5360.350098,0.094415,development
1,2007-11-21,5778.799805,5790.049805,5530.850098,5561.049805,-0.038030,2007-11-22,5564.649902,-0.008141,2007-11-22,5564.649902,0.030020,2007-11-22,5564.649902,0.009506,2007-11-22,5564.649902,0.067453,development
2,2007-12-17,6037.950195,6039.950195,5740.600098,5777.000000,-0.044761,2007-12-18,5777.600098,-0.006110,2007-12-18,5777.600098,-0.001921,2007-12-18,5777.600098,0.050739,2007-12-18,5777.600098,0.069544,development
3,2008-01-18,5907.750000,5908.750000,5677.000000,5705.299805,-0.035159,2008-01-21,5705.000000,-0.086976,2008-01-21,5705.000000,-0.087923,2008-01-21,5705.000000,-0.056380,2008-01-21,5705.000000,-0.067967,development
4,2008-02-07,5322.549805,5344.600098,5113.850098,5133.250000,-0.035566,2008-02-08,5132.100098,-0.002290,2008-02-08,5132.100098,-0.057257,2008-02-08,5132.100098,0.013620,2008-02-08,5132.100098,0.011633,development


In [22]:
df[
    (df["Date"] >= "2007-10-18") &
    (df["Date"] <= "2007-10-30")
][
    ["Date", "Open", "High", "Low", "Close"]
]

,Date,Open,High,Low,Close
22,2007-10-18,5551.100098,5736.799805,5269.649902,5351.000000
23,2007-10-19,5360.350098,5390.850098,5101.750000,5215.299805
24,2007-10-22,5202.750000,5247.399902,5070.899902,5184.000000
25,2007-10-23,5185.299805,5488.500000,5176.850098,5473.700195
26,2007-10-24,5477.600098,5577.899902,5419.399902,5496.149902
27,2007-10-25,5499.049805,5605.950195,5469.299805,5568.950195
28,2007-10-26,5564.250000,5716.899902,5513.350098,5702.299805
29,2007-10-29,5708.899902,5922.500000,5708.899902,5905.899902
30,2007-10-30,5917.549805,5976.000000,5833.899902,5868.750000


In [23]:
event = events.iloc[0]

event[
    [
        "Date",
        "event_return",
        "entry_date_1",
        "entry_price_1",
        "forward_return_1",
        "forward_return_3",
        "forward_return_5",
        "forward_return_10",
    ]
]

Date                 2007-10-18 00:00:00
event_return                   -0.037469
entry_date_1         2007-10-19 00:00:00
entry_price_1                5360.350098
forward_return_1                -0.02706
forward_return_3                0.021146
forward_return_5                0.038915
forward_return_10               0.094415
Name: 0, dtype: object

In [24]:
event = events.iloc[0]

event[
    [
        "Date",
        "event_return",
        "entry_date_1",
        "entry_price_1",
        "forward_return_1",
        "forward_return_3",
        "forward_return_5",
        "forward_return_10",
    ]
]

Date                 2007-10-18 00:00:00
event_return                   -0.037469
entry_date_1         2007-10-19 00:00:00
entry_price_1                5360.350098
forward_return_1                -0.02706
forward_return_3                0.021146
forward_return_5                0.038915
forward_return_10               0.094415
Name: 0, dtype: object

In [25]:
df.loc[
    event.name:event.name + 10,
    ["Date", "Open", "High", "Low", "Close"]
]

,Date,Open,High,Low,Close
0,2007-09-17,4518.450195,4549.049805,4482.850098,4494.649902
1,2007-09-18,4494.100098,4551.799805,4481.549805,4546.200195
2,2007-09-19,4550.250000,4739.000000,4550.250000,4732.350098
3,2007-09-20,4734.850098,4760.850098,4721.149902,4747.549805
4,2007-09-21,4752.950195,4855.700195,4733.700195,4837.549805
5,2007-09-24,4837.149902,4941.149902,4837.149902,4932.200195
6,2007-09-25,4939.100098,4953.899902,4878.149902,4938.850098
7,2007-09-26,4937.600098,4980.850098,4930.350098,4940.500000
8,2007-09-27,4942.700195,5016.399902,4942.700195,5000.549805
9,2007-09-28,4996.450195,5055.799805,4996.450195,5021.350098


1-Day

In [26]:
entry = event["entry_price_1"]

actual = (
    df.loc[event.name + 1, "Close"] / entry
) - 1

print(actual)
print(event["forward_return_1"])

-0.1518837179496405
-0.02705985436141034


3-Day

In [27]:
entry = event["entry_price_1"]

actual = (
    df.loc[event.name + 1, "Close"] / entry
) - 1

print(actual)
print(event["forward_return_1"])

-0.1518837179496405
-0.02705985436141034


5-Day

In [28]:
actual = (
    df.loc[event.name + 3, "Close"] /
    event["entry_price_1"]
) - 1

print(actual)
print(event["forward_return_3"])

-0.11432094579730712
0.02114602509000507


10-Day

In [29]:
actual = (
    df.loc[event.name + 10, "Close"] /
    event["entry_price_1"]
) - 1

print(actual)
print(event["forward_return_10"])

-0.05436210266772712
0.09441549309951536


In [30]:
events.shape

(53, 19)

In [31]:
events["period"].value_counts()

period
development    38
oos            15
Name: count, dtype: int64

In [32]:
events[
    [
        "Date",
        "event_return",
        "forward_return_1",
        "forward_return_3",
        "forward_return_5",
        "forward_return_10",
        "period",
    ]
].head(10)

,Date,event_return,forward_return_1,forward_return_3,forward_return_5,forward_return_10,period
0,2007-10-18,-0.037469,-0.027060,0.021146,0.038915,0.094415,development
1,2007-11-21,-0.038030,-0.008141,0.030020,0.009506,0.067453,development
2,2007-12-17,-0.044761,-0.006110,-0.001921,0.050739,0.069544,development
3,2008-01-18,-0.035159,-0.086976,-0.087923,-0.056380,-0.067967,development
4,2008-02-07,-0.035566,-0.002290,-0.057257,0.013620,0.011633,development
5,2008-03-03,-0.051785,-0.019018,-0.037702,-0.018685,-0.085821,development
6,2008-03-13,-0.050985,0.026385,-0.019637,-0.003017,0.023941,development
7,2008-03-31,-0.041987,0.000824,0.007591,0.005395,0.030408,development
8,2008-06-20,-0.034789,-0.019478,-0.022638,-0.049297,-0.077026,development
9,2008-07-01,-0.035589,0.050843,0.030986,0.023939,-0.008780,development


In [33]:
from src.baseline import calculate_baseline_returns

In [34]:
baseline = calculate_baseline_returns(
    df, holding_periods=(1,3,5,10),
)
baseline.head()

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10
0,2007-09-17,0.011593,0.056396,0.097483,0.127912
1,2007-09-18,0.040020,0.063139,0.085402,0.145168
2,2007-09-19,0.002682,0.041680,0.043433,0.100066
3,2007-09-20,0.017799,0.039113,0.052094,0.091080
4,2007-09-21,0.019650,0.021366,0.038080,0.051260


In [35]:
baseline.shape

(4585, 5)

In [36]:
baseline[
    [
        "Date",
        "baseline_return_1",
        "baseline_return_3",
        "baseline_return_5",
        "baseline_return_10",
    ]
].head(10)

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10
0,2007-09-17,0.011593,0.056396,0.097483,0.127912
1,2007-09-18,0.040020,0.063139,0.085402,0.145168
2,2007-09-19,0.002682,0.041680,0.043433,0.100066
3,2007-09-20,0.017799,0.039113,0.052094,0.091080
4,2007-09-21,0.019650,0.021366,0.038080,0.051260
5,2007-09-24,-0.000051,0.012441,0.026290,0.078587
6,2007-09-25,0.000587,0.016962,0.055330,0.102044
7,2007-09-26,0.011704,0.025543,0.053807,0.117780
8,2007-09-27,0.004984,0.042900,0.037907,0.086421
9,2007-09-28,0.009449,0.037270,0.012666,0.129224


In [37]:
baseline[baseline["Date"] == pd.Timestamp("2007-10-18")]

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10
22,2007-10-18,-0.02706,0.021146,0.038915,0.094415


Create the clean baseline population

In [38]:
event_dates = set(
    pd.to_datetime(events["Date"])
)

baseline["Date"] = pd.to_datetime(
    baseline["Date"]
)

baseline_non_event = baseline[
    ~baseline["Date"].isin(event_dates)
].copy()

baseline_non_event = baseline_non_event.reset_index(
    drop=True
)

In [39]:
baseline_non_event.shape

(4532, 5)

In [40]:
baseline_non_event.head()

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10
0,2007-09-17,0.011593,0.056396,0.097483,0.127912
1,2007-09-18,0.040020,0.063139,0.085402,0.145168
2,2007-09-19,0.002682,0.041680,0.043433,0.100066
3,2007-09-20,0.017799,0.039113,0.052094,0.091080
4,2007-09-21,0.019650,0.021366,0.038080,0.051260


Add the same development/OOS split

In [41]:
baseline_non_event["period"] = np.where(
    baseline_non_event["Date"] < pd.Timestamp("2020-01-01"),
    "development", "oos",
)

In [42]:
baseline_non_event["period"].value_counts()

period
development    2963
oos            1569
Name: count, dtype: int64

First descriptive comparison

In [43]:
horizons = [1, 3, 5, 10]

event_summary = {}

for h in horizons:
    returns = events[
        f"forward_return_{h}"
    ].dropna()

    event_summary[h] = {
        "n": len(returns),
        "mean": returns.mean(),
        "median": returns.median(),
        "win_rate": (returns > 0).mean(),
        "std": returns.std(),
    }

event_summary

{1: {'n': 53,
  'mean': np.float64(-0.0002900340570973539),
  'median': np.float64(0.0005769980693794974),
  'win_rate': np.float64(0.5283018867924528),
  'std': np.float64(0.024593293392500406)},
 3: {'n': 53,
  'mean': np.float64(0.0017272705743365405),
  'median': np.float64(0.006782467466005526),
  'win_rate': np.float64(0.5849056603773585),
  'std': np.float64(0.04411293695580173)},
 5: {'n': 53,
  'mean': np.float64(0.0074232859972477114),
  'median': np.float64(0.013620136204216315),
  'win_rate': np.float64(0.6037735849056604),
  'std': np.float64(0.04858368612414845)},
 10: {'n': 53,
  'mean': np.float64(0.003312384597923094),
  'median': np.float64(0.015439551096752213),
  'win_rate': np.float64(0.5849056603773585),
  'std': np.float64(0.08359437014274604)}}

In [44]:
baseline_summary = {}

for h in horizons:
    returns = baseline_non_event[
        f"baseline_return_{h}"
    ].dropna()

    baseline_summary[h] = {
        "n": len(returns), "mean": returns.mean(),
        "median": returns.median(), "std": returns.std(),
        "win_rate": (returns > 0).mean(),
    }

baseline_summary

{1: {'n': 4531,
  'mean': np.float64(-0.0005020541394949192),
  'median': np.float64(-0.00046557536931701726),
  'std': np.float64(0.011540635771099492),
  'win_rate': np.float64(0.4747296402560141)},
 3: {'n': 4529,
  'mean': np.float64(0.00037346212555367696),
  'median': np.float64(0.001031978970019054),
  'std': np.float64(0.021617777002603308),
  'win_rate': np.float64(0.526164716272908)},
 5: {'n': 4527,
  'mean': np.float64(0.0011887662208082256),
  'median': np.float64(0.002206488531109052),
  'std': np.float64(0.02826865093858146),
  'win_rate': np.float64(0.5420808482438702)},
 10: {'n': 4522,
  'mean': np.float64(0.0033744152431436645),
  'median': np.float64(0.004832403331914148),
  'std': np.float64(0.03922548898693917),
  'win_rate': np.float64(0.5616983635559487)}}

Put them into one table

In [45]:
comparison_rows = []

for h in horizons:
    event_returns = events[
        f"forward_return_{h}"
    ].dropna()

    baseline_returns = baseline_non_event[
        f"baseline_return_{h}"
    ].dropna()

    comparison_rows.append({
        "horizon": h,

        "event_n": len(event_returns),
        "event_mean": event_returns.mean(),
        "event_median": event_returns.median(),
        "event_win_rate": (event_returns > 0).mean(),

        "baseline_n": len(baseline_returns),
        "baseline_mean": baseline_returns.mean(),
        "baseline_median": baseline_returns.median(),
        "baseline_win_rate": (baseline_returns > 0).mean(),

        "mean_difference": (
            event_returns.mean()
            - baseline_returns.mean()
        ),
    })

comparison = pd.DataFrame(
    comparison_rows
)

comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,53,-0.000290,0.000577,0.528302,4531,-0.000502,-0.000466,0.474730,0.000212
1,3,53,0.001727,0.006782,0.584906,4529,0.000373,0.001032,0.526165,0.001354
2,5,53,0.007423,0.013620,0.603774,4527,0.001189,0.002206,0.542081,0.006235
3,10,53,0.003312,0.015440,0.584906,4522,0.003374,0.004832,0.561698,-0.000062


Run it for every horizon

In [46]:
from src.statistics import bootstrap_mean_ci, bootstrap_mean_difference_ci

bootstrap_results = []

for h in [1, 3, 5, 10]:

    values = events[
        f"forward_return_{h}"
    ].dropna()

    result = bootstrap_mean_ci(values)

    bootstrap_results.append({
        "horizon": h,
        **result,
    })

bootstrap_ci = pd.DataFrame(
    bootstrap_results
)

bootstrap_ci

,horizon,mean,ci_lower,ci_upper,n
0,1,-0.000290,-0.006946,0.006144,53
1,3,0.001727,-0.010213,0.013084,53
2,5,0.007423,-0.005639,0.020451,53
3,10,0.003312,-0.020047,0.024868,53


In [47]:
difference_results = []

for h in [1, 3, 5, 10]:

    event_values = events[
        f"forward_return_{h}"
    ].dropna()

    baseline_values = baseline_non_event[
        f"baseline_return_{h}"
    ].dropna()

    result = bootstrap_mean_difference_ci(
        event_values,
        baseline_values,
    )

    difference_results.append({
        "horizon": h,
        **result,
    })

difference_ci = pd.DataFrame(
    difference_results
)

difference_ci

,horizon,difference,ci_lower,ci_upper
0,1,0.000212,-0.006407,0.006709
1,3,0.001354,-0.011171,0.012577
2,5,0.006235,-0.006690,0.019260
3,10,-0.000062,-0.023338,0.021538


In [48]:
oos_events = events[
    events["period"] == "oos"
].copy()

oos_baseline = baseline_non_event[
    baseline_non_event["period"] == "oos"
].copy()

In [49]:
oos_rows = []

for h in [1,3,5,10]:
    event_returns = oos_events[
        f"forward_return_{h}"
    ].dropna()

    baseline_returns = oos_baseline[
        f"baseline_return_{h}"
    ].dropna()

    oos_rows.append({
        "horizon": h,
        "event_n": len(event_returns),
        "event_mean": event_returns.mean(),
        "event_median": event_returns.median(),
        "event_win_rate": (
            event_returns > 0
        ).mean(),

        "baseline_n": len(baseline_returns),
        "baseline_mean": baseline_returns.mean(),
        "baseline_median": baseline_returns.median(),
        "baseline_win_rate": (
            baseline_returns > 0
        ).mean(),
        "mean_difference": (
            event_returns.mean() - baseline_returns.mean()
        ),
    })

oos_comparison = pd.DataFrame(oos_rows)

oos_comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,15,0.005853,0.008019,0.800000,1568,-0.000590,-0.000243,0.483418,0.006443
1,3,15,0.001316,0.014141,0.600000,1566,0.000416,0.001198,0.533206,0.000900
2,5,15,0.003491,0.016024,0.600000,1564,0.001412,0.002273,0.539642,0.002079
3,10,15,0.000207,0.023626,0.733333,1559,0.003926,0.005032,0.569596,-0.003719


In [50]:
development_events = events[
    events["period"] == "development"
].copy()

development_baseline = baseline_non_event[
    baseline_non_event["period"] == "development"
].copy()

In [51]:
development_rows = []

for h in [1,3,5,10]:
    event_returns = development_events[
        f"forward_return_{h}"
    ].dropna()

    baseline_returns = development_baseline[
        f"baseline_return_{h}"
    ].dropna()

    development_rows.append({
        "horizon": h,
        "event_n": len(event_returns),
        "event_mean": event_returns.mean(),
        "event_median": event_returns.median(),
        "event_win_rate": (
            event_returns > 0
        ).mean(),

        "baseline_n": len(baseline_returns),
        "baseline_mean": baseline_returns.mean(),
        "baseline_median": baseline_returns.median(),
        "baseline_win_rate": (
            baseline_returns > 0
        ).mean(),
        "mean_difference": (
            event_returns.mean() - baseline_returns.mean()
        ),
    })

development_comparison = pd.DataFrame(development_rows)

development_comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,38,-0.002715,-0.002076,0.421053,2963,-0.000456,-0.000576,0.470132,-0.002259
1,3,38,0.001889,0.005385,0.578947,2963,0.000351,0.000957,0.522443,0.001539
2,5,38,0.008975,0.013162,0.605263,2963,0.001071,0.002182,0.543368,0.007905
3,10,38,0.004538,0.007283,0.526316,2963,0.003084,0.004810,0.557543,0.001454


Now test OOS uncertainty

In [52]:
oos_difference_results = []

for h in [1, 3, 5, 10]:

    event_values = oos_events[
        f"forward_return_{h}"
    ].dropna()

    baseline_values = oos_baseline[
        f"baseline_return_{h}"
    ].dropna()

    result = bootstrap_mean_difference_ci(
        event_values,
        baseline_values,
        n_bootstrap=10_000,
        random_state=42,
    )

    oos_difference_results.append({
        "horizon": h,
        **result,
    })

oos_difference_ci = pd.DataFrame(
    oos_difference_results
)

oos_difference_ci

,horizon,difference,ci_lower,ci_upper
0,1,0.006443,-0.001175,0.013376
1,3,0.000900,-0.015203,0.016305
2,5,0.002079,-0.022899,0.021892
3,10,-0.003719,-0.056803,0.040173


In [53]:
from src.baseline import calculate_baseline_returns, create_strict_baseline

Create the Strict baseline

In [54]:
baseline_strict = create_strict_baseline(
    df=df, events=events, 
    baseline=baseline, window=5,
)

In [55]:
baseline_strict.shape

(4267, 5)

Add the development/OOS labels

In [56]:
baseline_strict["period"] = np.where(
    baseline_strict["Date"] < pd.Timestamp("2020-01-01"),
    "development", "oos",
)

In [57]:
baseline_strict["period"].value_counts()

period
development    2773
oos            1494
Name: count, dtype: int64

Verify the exclusion logic

In [58]:
# Pick the first event:
first_event = events.iloc[0]

first_event["Date"]

Timestamp('2007-10-18 00:00:00')

Now inspect the next six trading observations in df:

In [59]:
event_position = first_event.name

df.loc[
    event_position:event_position + 5,
    ["Date", "Open", "Close"]
]

,Date,Open,Close
0,2007-09-17,4518.450195,4494.649902
1,2007-09-18,4494.100098,4546.200195
2,2007-09-19,4550.250000,4732.350098
3,2007-09-20,4734.850098,4747.549805
4,2007-09-21,4752.950195,4837.549805
5,2007-09-24,4837.149902,4932.200195


Now inspect the next six trading observations in df:

In [60]:
excluded_check = baseline_strict[
    baseline_strict["Date"].isin(
        df.loc[
            event_position:event_position + 5,
            "Date"
        ]
    )
]

excluded_check

,Date,baseline_return_1,baseline_return_3,baseline_return_5,baseline_return_10,period
0,2007-09-17,0.011593,0.056396,0.097483,0.127912,development
1,2007-09-18,0.040020,0.063139,0.085402,0.145168,development
2,2007-09-19,0.002682,0.041680,0.043433,0.100066,development
3,2007-09-20,0.017799,0.039113,0.052094,0.091080,development
4,2007-09-21,0.019650,0.021366,0.038080,0.051260,development
5,2007-09-24,-0.000051,0.012441,0.026290,0.078587,development


Verify we're not accidentally deleting unrelated observations

The strict baseline should only remove windows associated with actual selected events.

In [61]:
print("Original baseline:", len(baseline_non_event))
print("Strict baseline:", len(baseline_strict))
print(
    "Removed:", len(baseline_non_event) - len(baseline_strict)
)
# This tells us how many additional observations were removed by the strict specification.

Original baseline: 4532
Strict baseline: 4267
Removed: 265


**Recalculate the descriptive comparison**

Now compare events against the strict baseline.

In [62]:
strict_rows = []

for h in [1,3,5,10]:
    event_returns = events[
        f"forward_return_{h}"
    ].dropna()

    baseline_returns = baseline_strict[
        f"baseline_return_{h}"
    ].dropna()

    strict_rows.append({
        "horizon": h, "event_n": len(event_returns),
        "event_mean": event_returns.mean(),
        "event_median": event_returns.median(),
        "event_win_rate": (event_returns > 0).mean(),

        "baseline_n": len(baseline_returns),
        "baseline_mean": baseline_returns.mean(),
        "baseline_median": baseline_returns.median(),
        "baseline_win_rate": (baseline_returns > 0).mean(),

        "mean_difference": (event_returns.mean() - baseline_returns.mean()),
    })

strict_comparison = pd.DataFrame(strict_rows)

strict_comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,53,-0.000290,0.000577,0.528302,4266,-0.000698,-0.000547,0.467886,0.000408
1,3,53,0.001727,0.006782,0.584906,4264,-0.000092,0.000835,0.521811,0.001819
2,5,53,0.007423,0.013620,0.603774,4262,0.000815,0.001987,0.538949,0.006608
3,10,53,0.003312,0.015440,0.584906,4257,0.002702,0.004402,0.557670,0.000611


**Compare the two baseline specifications**

In [63]:
baseline_comparison = comparison[
    [
        "horizon",
        "baseline_mean",
        "mean_difference",
    ]
].merge(
    strict_comparison[
        [
            "horizon",
            "baseline_mean",
            "mean_difference",
        ]
    ],
    on="horizon",
    suffixes=(
        "_original",
        "_strict",
    ),
)

baseline_comparison

,horizon,baseline_mean_original,mean_difference_original,baseline_mean_strict,mean_difference_strict
0,1,-0.000502,0.000212,-0.000698,0.000408
1,3,0.000373,0.001354,-0.000092,0.001819
2,5,0.001189,0.006235,0.000815,0.006608
3,10,0.003374,-0.000062,0.002702,0.000611


In [64]:
baseline_strict.shape

(4267, 6)

In [65]:
baseline_strict["period"].value_counts()

period
development    2773
oos            1494
Name: count, dtype: int64

In [66]:
strict_comparison

,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
0,1,53,-0.000290,0.000577,0.528302,4266,-0.000698,-0.000547,0.467886,0.000408
1,3,53,0.001727,0.006782,0.584906,4264,-0.000092,0.000835,0.521811,0.001819
2,5,53,0.007423,0.013620,0.603774,4262,0.000815,0.001987,0.538949,0.006608
3,10,53,0.003312,0.015440,0.584906,4257,0.002702,0.004402,0.557670,0.000611


In [67]:
baseline_comparison

,horizon,baseline_mean_original,mean_difference_original,baseline_mean_strict,mean_difference_strict
0,1,-0.000502,0.000212,-0.000698,0.000408
1,3,0.000373,0.001354,-0.000092,0.001819
2,5,0.001189,0.006235,0.000815,0.006608
3,10,0.003374,-0.000062,0.002702,0.000611


In [68]:
from src.events import detect_events, calculate_forward_returns

thresholds = [
    -0.02,
    -0.025,
    -0.03,
    -0.035,
    -0.04,
    -0.05,
]

threshold_counts = []

for threshold in thresholds:
    threshold_events = detect_events(
        df=df,
        threshold=threshold,
        overlap_window=5,
    )

    threshold_counts.append({
        "threshold": threshold,
        "event_count": len(threshold_events),
    })

threshold_counts = pd.DataFrame(
    threshold_counts
)

threshold_counts

,threshold,event_count
0,-0.020,123
1,-0.025,75
2,-0.030,53
3,-0.035,35
4,-0.040,26
5,-0.050,15


**Create a reusable threshold-analysis function**

In [69]:
def calculate_threshold_analysis(
    df,
    threshold,
    overlap_window=5,
    strict_window=5,
):
    # --------------------------------------------------
    # 1. Detect events
    # --------------------------------------------------

    threshold_events = detect_events(
        df=df,
        threshold=threshold,
        overlap_window=overlap_window,
    )

    # --------------------------------------------------
    # 2. Calculate forward returns
    # --------------------------------------------------

    threshold_events = calculate_forward_returns(
        df=df,
        events=threshold_events,
        holding_periods=(1, 3, 5, 10),
    )

    # --------------------------------------------------
    # 3. Calculate baseline
    # --------------------------------------------------

    threshold_baseline = (
        calculate_baseline_returns(df)
    )

    # --------------------------------------------------
    # 4. Strict baseline
    # --------------------------------------------------

    threshold_baseline_strict = (
        create_strict_baseline(
            df=df,
            events=threshold_events,
            baseline=threshold_baseline,
            window=strict_window,
        )
    )

    # --------------------------------------------------
    # 5. Development / OOS labels
    # --------------------------------------------------

    threshold_events = threshold_events.copy()

    threshold_baseline_strict = (
        threshold_baseline_strict.copy()
    )

    threshold_events["period"] = np.where(
        threshold_events["Date"]
        < pd.Timestamp("2020-01-01"),
        "development",
        "oos",
    )

    threshold_baseline_strict["period"] = np.where(
        threshold_baseline_strict["Date"]
        < pd.Timestamp("2020-01-01"),
        "development",
        "oos",
    )

    # --------------------------------------------------
    # 6. Compare event vs baseline
    # --------------------------------------------------

    rows = []

    for period in [
        "development",
        "oos",
    ]:

        period_events = threshold_events[
            threshold_events["period"] == period
        ]

        period_baseline = threshold_baseline_strict[
            threshold_baseline_strict["period"]
            == period
        ]

        for h in [1, 3, 5, 10]:

            event_returns = period_events[
                f"forward_return_{h}"
            ].dropna()

            baseline_returns = period_baseline[
                f"baseline_return_{h}"
            ].dropna()

            rows.append({
                "threshold": threshold,
                "period": period,
                "horizon": h,

                "event_n": len(
                    event_returns
                ),

                "event_mean": (
                    event_returns.mean()
                    if len(event_returns)
                    else np.nan
                ),

                "event_median": (
                    event_returns.median()
                    if len(event_returns)
                    else np.nan
                ),

                "event_win_rate": (
                    (event_returns > 0).mean()
                    if len(event_returns)
                    else np.nan
                ),

                "baseline_n": len(
                    baseline_returns
                ),

                "baseline_mean": (
                    baseline_returns.mean()
                    if len(baseline_returns)
                    else np.nan
                ),

                "baseline_median": (
                    baseline_returns.median()
                    if len(baseline_returns)
                    else np.nan
                ),

                "baseline_win_rate": (
                    (baseline_returns > 0).mean()
                    if len(baseline_returns)
                    else np.nan
                ),

                "mean_difference": (
                    event_returns.mean()
                    - baseline_returns.mean()
                    if (
                        len(event_returns)
                        and len(baseline_returns)
                    )
                    else np.nan
                ),
            })

    return (
        threshold_events,
        threshold_baseline_strict,
        pd.DataFrame(rows),
    )

**Run all thresholds**

In [70]:
thresholds = [
    -0.02, -0.025, -0.03,
    -0.035, -0.04, -0.05,
]

threshold_results = []
threshold_datasets = {}

for threshold in thresholds:
    (
        threshold_events,
        threshold_baseline_strict,
        threshold_result,
    ) = calculate_threshold_analysis(
        df=df, threshold=threshold,
    )

    threshold_results.append(
        threshold_result
    )

    threshold_datasets[threshold] = {
        "events": threshold_events,
        "baseline": threshold_baseline_strict,
    }

threshold_analysis = pd.concat(
    threshold_results,
    ignore_index=True,
)

**First inspect event counts by period**

In [71]:
threshold_event_counts = (
    threshold_analysis[
        [
            "threshold",
            "period",
            "event_n",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["threshold", "period"]
    )
)

threshold_event_counts

,threshold,period,event_n
40,-0.050,development,11
44,-0.050,oos,4
32,-0.040,development,20
36,-0.040,oos,6
24,-0.035,development,26
28,-0.035,oos,9
16,-0.030,development,38
20,-0.030,oos,15
8,-0.025,development,54
12,-0.025,oos,21


**Generate the main threshold table**

In [72]:
threshold_summary = (
    threshold_analysis[
        [
            "threshold",
            "period",
            "horizon",
            "event_n",
            "event_mean",
            "baseline_mean",
            "mean_difference",
            "event_win_rate",
            "baseline_win_rate",
        ]
    ]
    .sort_values(
        [
            "period",
            "threshold",
            "horizon",
        ]
    )
)

threshold_summary

,threshold,period,horizon,event_n,event_mean,baseline_mean,mean_difference,event_win_rate,baseline_win_rate
40,-0.050,development,1,11,-0.012855,-0.000429,-0.012426,0.181818,0.469847
41,-0.050,development,3,11,-0.016485,0.000527,-0.017012,0.272727,0.525043
42,-0.050,development,5,11,-0.001600,0.001431,-0.003030,0.363636,0.547530
43,-0.050,development,10,11,-0.012877,0.003537,-0.016414,0.454545,0.560136
32,-0.040,development,1,20,-0.009570,-0.000592,-0.008978,0.250000,0.465810
33,-0.040,development,3,20,0.002539,0.000111,0.002428,0.500000,0.521000
34,-0.040,development,5,20,0.017655,0.000987,0.016668,0.500000,0.545297
35,-0.040,development,10,20,0.006126,0.002909,0.003217,0.550000,0.554669
24,-0.035,development,1,26,-0.013405,-0.000591,-0.012814,0.307692,0.464323
25,-0.035,development,3,26,-0.009554,-0.000130,-0.009424,0.461538,0.518453


**Create an easier-to-read 5-day table**

In [73]:
threshold_5d = (
    threshold_analysis[
        threshold_analysis["horizon"] == 5
    ][
        [
            "threshold",
            "period",
            "event_n",
            "event_mean",
            "baseline_mean",
            "mean_difference",
            "event_win_rate",
            "baseline_win_rate",
        ]
    ]
    .sort_values(
        ["period", "threshold"]
    )
)

threshold_5d

,threshold,period,event_n,event_mean,baseline_mean,mean_difference,event_win_rate,baseline_win_rate
42,-0.050,development,11,-0.001600,0.001431,-0.003030,0.363636,0.547530
34,-0.040,development,20,0.017655,0.000987,0.016668,0.500000,0.545297
26,-0.035,development,26,0.010128,0.000586,0.009542,0.538462,0.541301
18,-0.030,development,38,0.008975,0.000446,0.008530,0.605263,0.539488
10,-0.025,development,54,0.002134,0.000274,0.001859,0.574074,0.534554
2,-0.020,development,92,0.000820,0.000636,0.000185,0.500000,0.540220
46,-0.050,oos,4,-0.001588,0.001631,-0.003219,0.500000,0.540193
38,-0.040,oos,6,-0.010031,0.001602,-0.011633,0.500000,0.539857
30,-0.035,oos,9,-0.009319,0.001948,-0.011267,0.444444,0.540984
22,-0.030,oos,15,0.003491,0.001504,0.001987,0.600000,0.537945


**Don't select a winner**

In [74]:
threshold_analysis.sort_values(
    "mean_difference",
    ascending=False
)

,threshold,period,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
45,-0.050,oos,3,4,0.028425,0.018535,0.500000,1557,0.000585,0.001321,0.535003,0.027841
44,-0.050,oos,1,4,0.021386,0.008107,0.500000,1559,-0.000588,-0.000162,0.486851,0.021974
34,-0.040,development,5,20,0.017655,0.001189,0.500000,2881,0.000987,0.002261,0.545297,0.016668
47,-0.050,oos,10,4,0.015494,0.005763,0.500000,1550,0.003833,0.005100,0.572258,0.011661
11,-0.025,development,10,54,0.011869,0.020327,0.574074,2677,0.001561,0.003925,0.549869,0.010308
26,-0.035,development,5,26,0.010128,0.007451,0.538462,2845,0.000586,0.002037,0.541301,0.009542
36,-0.040,oos,1,6,0.008666,0.010326,0.833333,1547,-0.000574,-0.000177,0.486102,0.009239
18,-0.030,development,5,38,0.008975,0.013162,0.605263,2773,0.000446,0.001980,0.539488,0.008530
12,-0.025,oos,1,21,0.006210,0.008019,0.761905,1457,-0.000642,-0.000296,0.478380,0.006852
20,-0.030,oos,1,15,0.005853,0.008019,0.800000,1493,-0.000673,-0.000280,0.480241,0.006526


In [75]:
from src.statistics import bootstrap_mean_difference

**Get the primary -3% datasets**

In [76]:
threshold_datasets

{-0.02: {'events':           Date          Open          High           Low         Close  \
  0   2007-10-18   5551.100098   5736.799805   5269.649902   5351.000000   
  1   2007-11-20   5911.250000   5923.700195   5755.799805   5780.899902   
  2   2007-12-17   6037.950195   6039.950195   5740.600098   5777.000000   
  3   2008-01-15   6226.350098   6260.450195   6053.299805   6074.250000   
  4   2008-01-24   5208.000000   5357.200195   4995.799805   5033.450195   
  ..         ...           ...           ...           ...           ...   
  118 2024-08-05  24302.849609  24350.050781  23893.699219  24055.599609   
  119 2024-10-03  25452.849609  25639.449219  25230.300781  25250.099609   
  120 2025-04-07  21758.400391  22254.000000  21743.650391  22161.599609   
  121 2026-03-13  23462.500000  23492.400391  23112.000000  23151.099609   
  122 2026-03-23  22824.349609  22851.699219  22471.250000  22512.650391   
  
       event_return entry_date_1  entry_price_1  forward_return_1 en

In [77]:
primary_events = threshold_datasets[-0.03]["events"]

primary_baseline = threshold_datasets[-0.03]["baseline"]

In [78]:
primary_events.shape

(53, 19)

In [79]:
primary_events["period"].value_counts()

period
development    38
oos            15
Name: count, dtype: int64

**Run bootstrap for development and OOS**

In [80]:
bootstrap_rows = []

for period in [
    "development",
    "oos",
]:

    period_events = primary_events[
        primary_events["period"] == period
    ]

    period_baseline = primary_baseline[
        primary_baseline["period"] == period
    ]

    for h in [1, 3, 5, 10]:

        event_returns = period_events[
            f"forward_return_{h}"
        ].dropna()

        baseline_returns = period_baseline[
            f"baseline_return_{h}"
        ].dropna()

        result = bootstrap_mean_difference(
            event_returns=event_returns,
            baseline_returns=baseline_returns,
            n_bootstrap=10_000,
            confidence=0.95,
            random_state=42,
        )

        bootstrap_rows.append({
            "threshold": -0.03,
            "period": period,
            "horizon": h,
            **result,
        })

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

**Inspect the Result**

In [81]:
bootstrap_results

,threshold,period,horizon,event_n,baseline_n,observed_difference,bootstrap_se,ci_lower,ci_upper,p_value
0,-0.03,development,1,38.0,2773.0,-0.002003,0.004384,-0.010615,0.006730,0.6782
1,-0.03,development,3,38.0,2773.0,0.002233,0.007825,-0.013912,0.016780,0.7778
2,-0.03,development,5,38.0,2773.0,0.008530,0.007970,-0.007182,0.023974,0.5181
3,-0.03,development,10,38.0,2773.0,0.002521,0.012342,-0.022439,0.026704,0.8364
4,-0.03,oos,1,15.0,1493.0,0.006526,0.003727,-0.001005,0.013515,0.5171
5,-0.03,oos,3,15.0,1491.0,0.000940,0.008060,-0.015545,0.016401,0.9089
6,-0.03,oos,5,15.0,1489.0,0.001987,0.011564,-0.023013,0.021748,0.8727
7,-0.03,oos,10,15.0,1484.0,-0.003773,0.025091,-0.056700,0.040649,0.8814


Format returns as percentages

In [82]:
bootstrap_display = bootstrap_results.copy()

for col in [
    "observed_difference",
    "bootstrap_se",
    "ci_lower", "ci_upper",
]:
    bootstrap_display[col] *= 100

bootstrap_display

,threshold,period,horizon,event_n,baseline_n,observed_difference,bootstrap_se,ci_lower,ci_upper,p_value
0,-0.03,development,1,38.0,2773.0,-0.200331,0.438353,-1.061487,0.673042,0.6782
1,-0.03,development,3,38.0,2773.0,0.223276,0.782478,-1.391206,1.678003,0.7778
2,-0.03,development,5,38.0,2773.0,0.852988,0.796971,-0.718248,2.397418,0.5181
3,-0.03,development,10,38.0,2773.0,0.252068,1.234167,-2.243858,2.670444,0.8364
4,-0.03,oos,1,15.0,1493.0,0.652648,0.372664,-0.100454,1.351525,0.5171
5,-0.03,oos,3,15.0,1491.0,0.093977,0.805975,-1.554524,1.640057,0.9089
6,-0.03,oos,5,15.0,1489.0,0.198698,1.156429,-2.301310,2.174775,0.8727
7,-0.03,oos,10,15.0,1484.0,-0.377286,2.509090,-5.669973,4.064940,0.8814


**Also calculate the raw event statistics**

In [83]:
primary_summary = (
    threshold_analysis[
        threshold_analysis["threshold"] == -0.03
    ]
    .sort_values(
        ["period", "horizon"]
    )
)

primary_summary

,threshold,period,horizon,event_n,event_mean,event_median,event_win_rate,baseline_n,baseline_mean,baseline_median,baseline_win_rate,mean_difference
16,-0.03,development,1,38,-0.002715,-0.002076,0.421053,2773,-0.000712,-0.000724,0.461233,-0.002003
17,-0.03,development,3,38,0.001889,0.005385,0.578947,2773,-0.000343,0.000693,0.516769,0.002233
18,-0.03,development,5,38,0.008975,0.013162,0.605263,2773,0.000446,0.001980,0.539488,0.008530
19,-0.03,development,10,38,0.004538,0.007283,0.526316,2773,0.002018,0.004122,0.552110,0.002521
20,-0.03,oos,1,15,0.005853,0.008019,0.800000,1493,-0.000673,-0.000280,0.480241,0.006526
21,-0.03,oos,3,15,0.001316,0.014141,0.600000,1491,0.000377,0.001140,0.531187,0.000940
22,-0.03,oos,5,15,0.003491,0.016024,0.600000,1489,0.001504,0.002036,0.537945,0.001987
23,-0.03,oos,10,15,0.000207,0.023626,0.733333,1484,0.003980,0.004772,0.568059,-0.003773


**First Debug: compare the exact arrays**

In [84]:
threshold = -0.03
period = "oos"
h = 5

period_events = primary_events[
    primary_events["period"] == period
]

period_baseline = primary_baseline[
    primary_baseline["period"] == period
]

event_returns = period_events[
    f"forward_return_{h}"
].dropna()

baseline_returns = period_baseline[
    f"baseline_return_{h}"
].dropna()

print("Event N:", len(event_returns))
print("Baseline N:", len(baseline_returns))

print(
    "Event mean:",
    event_returns.mean()
)

print(
    "Baseline mean:",
    baseline_returns.mean()
)

print(
    "Difference:",
    event_returns.mean()
    - baseline_returns.mean()
)

Event N: 15
Baseline N: 1489
Event mean: 0.003491122444900922
Baseline mean: 0.0015041444590257135
Difference: 0.0019869779858752083


Then Run

In [85]:
check = bootstrap_mean_difference(
    event_returns=event_returns,
    baseline_returns=baseline_returns,
    n_bootstrap=10_000,
    confidence=0.95,
    random_state=42,
)

check

{'event_n': 15.0,
 'baseline_n': 1489.0,
 'observed_difference': 0.0019869779858752083,
 'bootstrap_se': 0.011564290691808822,
 'ci_lower': -0.02301309809247876,
 'ci_upper': 0.02174775392543411,
 'p_value': 0.8727}

In [86]:
check["observed_difference"]

0.0019869779858752083

**make the pipeline internally consistent**

In [87]:
diagnostic = []

for period in ["development", "oos"]:

    period_events = primary_events[
        primary_events["period"] == period
    ]

    period_baseline = primary_baseline[
        primary_baseline["period"] == period
    ]

    for h in [1, 3, 5, 10]:

        event_returns = period_events[
            f"forward_return_{h}"
        ].dropna()

        baseline_returns = period_baseline[
            f"baseline_return_{h}"
        ].dropna()

        observed = (
            event_returns.mean()
            - baseline_returns.mean()
        )

        bootstrap = bootstrap_mean_difference(
            event_returns,
            baseline_returns,
            n_bootstrap=10_000,
            random_state=42,
        )

        diagnostic.append({
            "period": period,
            "horizon": h,
            "event_n": len(event_returns),
            "baseline_n": len(baseline_returns),
            "direct_difference": observed,
            "bootstrap_difference":
                bootstrap["observed_difference"],
            "difference_check":
                np.isclose(
                    observed,
                    bootstrap[
                        "observed_difference"
                    ],
                ),
        })

diagnostic_df = pd.DataFrame(
    diagnostic
)

diagnostic_df

,period,horizon,event_n,baseline_n,direct_difference,bootstrap_difference,difference_check
0,development,1,38,2773,-0.002003,-0.002003,True
1,development,3,38,2773,0.002233,0.002233,True
2,development,5,38,2773,0.008530,0.008530,True
3,development,10,38,2773,0.002521,0.002521,True
4,oos,1,15,1493,0.006526,0.006526,True
5,oos,3,15,1491,0.000940,0.000940,True
6,oos,5,15,1489,0.001987,0.001987,True
7,oos,10,15,1484,-0.003773,-0.003773,True
